# Amravati Municipal Corporation Property Tax Dataset - Data Cleaning

This notebook performs complete data cleaning for the **Amravati Municipal Corporation Property Tax Dashboard** project.

### Steps covered
1. Import required libraries  
2. Upload and read dataset  
3. Understand dataset structure  
4. Check missing/null values  
5. Handle missing values  
6. Remove duplicate records  
7. Clean text columns  
8. Convert date columns  
9. Validate numeric columns  
10. Create useful calculated columns  
11. Export cleaned dataset for Power BI / analysis


## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

## 2. Upload and Read Dataset

Run the cell below in Google Colab and upload your file:

`amravati_property_tax_final_complete_50k.csv`


In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# Read CSV file
file_name = 'amravati_property_tax_final_complete_50k.csv'
df = pd.read_csv(file_name)

# Show first 5 rows
df.head()

## 3. Dataset Overview

In [ ]:
# Dataset shape
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])

# Column names
print('
Column Names:')
print(df.columns.tolist())

In [ ]:
# Basic information about dataset
df.info()

In [ ]:
# Summary statistics for numeric columns
df.describe()

## 4. Check Missing / Null Values

In [ ]:
# Check missing values count and percentage
missing_summary = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df)) * 100
})

missing_summary.sort_values(by='Missing_Count', ascending=False)

## 5. Handle Missing Values

Even if the current dataset has no null values, this step makes the cleaning process safe and reusable.

### Strategy used
- Numeric columns: fill with `0` where suitable for financial values
- Category/Text columns: fill with `Unknown`
- Date columns: convert to datetime and keep invalid dates as `NaT`


In [ ]:
# Create a copy before cleaning
clean_df = df.copy()

# Define column groups based on this dataset
numeric_columns = [
    'Property_ID', 'Pincode', 'Year', 'Area_sqft', 'Property_Value',
    'Tax_Due', 'Tax_Paid', 'Outstanding_Amount', 'Penalty_Amount',
    'Rebate_Amount', 'Last_Payment_Year', 'Years_Due'
]

categorical_columns = [
    'Zone', 'Ward_Number', 'Area_Name', 'Property_Category', 'Property_SubType',
    'Owner_Type', 'Owner_Name', 'Occupancy_Status', 'Payment_Mode', 'Payment_Status'
]

# Fill missing numeric values with 0
for col in numeric_columns:
    if col in clean_df.columns:
        clean_df[col] = pd.to_numeric(clean_df[col], errors='coerce')
        clean_df[col] = clean_df[col].fillna(0)

# Fill missing categorical values with 'Unknown'
for col in categorical_columns:
    if col in clean_df.columns:
        clean_df[col] = clean_df[col].fillna('Unknown')

# Convert Payment_Date to datetime
if 'Payment_Date' in clean_df.columns:
    clean_df['Payment_Date'] = pd.to_datetime(clean_df['Payment_Date'], errors='coerce')

# Check missing values after handling
clean_df.isnull().sum()

## 6. Remove Duplicate Records

In [ ]:
# Check duplicate rows
print('Duplicate rows before cleaning:', clean_df.duplicated().sum())

# Remove duplicate rows
clean_df = clean_df.drop_duplicates()

print('Duplicate rows after cleaning:', clean_df.duplicated().sum())
print('Updated shape:', clean_df.shape)

## 7. Clean Text Columns

This step removes extra spaces and standardizes text values.

In [ ]:
# Strip extra spaces from all object/text columns
text_columns = clean_df.select_dtypes(include='object').columns

for col in text_columns:
    clean_df[col] = clean_df[col].astype(str).str.strip()

# Standardize selected categorical columns
for col in ['Zone', 'Ward_Number', 'Area_Name', 'Property_Category', 'Property_SubType',
            'Owner_Type', 'Occupancy_Status', 'Payment_Mode', 'Payment_Status']:
    if col in clean_df.columns:
        clean_df[col] = clean_df[col].str.title()

clean_df.head()

## 8. Fix Data Types

In [ ]:
# Convert integer columns safely
integer_columns = [
    'Property_ID', 'Pincode', 'Year', 'Area_sqft', 'Property_Value',
    'Tax_Due', 'Tax_Paid', 'Outstanding_Amount', 'Penalty_Amount',
    'Rebate_Amount', 'Last_Payment_Year', 'Years_Due'
]

for col in integer_columns:
    if col in clean_df.columns:
        clean_df[col] = pd.to_numeric(clean_df[col], errors='coerce').fillna(0).astype(int)

# Confirm data types
clean_df.dtypes

## 9. Validate Numeric Columns

This step checks whether important numeric columns contain negative or invalid values.

In [ ]:
# Check negative values in financial and area columns
check_columns = ['Area_sqft', 'Property_Value', 'Tax_Due', 'Tax_Paid', 'Outstanding_Amount', 'Penalty_Amount', 'Rebate_Amount']

for col in check_columns:
    if col in clean_df.columns:
        negative_count = (clean_df[col] < 0).sum()
        print(f'{col}: {negative_count} negative values')

In [ ]:
# Replace negative values with 0 for selected numeric columns
for col in check_columns:
    if col in clean_df.columns:
        clean_df[col] = clean_df[col].apply(lambda x: 0 if x < 0 else x)

print('Negative value handling completed.')

## 10. Validate Payment Status

This step checks whether payment status matches paid/outstanding amounts.

In [ ]:
# Unique payment status values
if 'Payment_Status' in clean_df.columns:
    print(clean_df['Payment_Status'].value_counts())

In [ ]:
# Create corrected payment status based on amount logic
# Fully Paid: Outstanding Amount = 0
# Partial Paid: Tax Paid > 0 and Outstanding Amount > 0
# Unpaid: Tax Paid = 0 and Outstanding Amount > 0

def define_payment_status(row):
    if row['Outstanding_Amount'] == 0:
        return 'Fully Paid'
    elif row['Tax_Paid'] > 0 and row['Outstanding_Amount'] > 0:
        return 'Partial Paid'
    else:
        return 'Unpaid'

if {'Tax_Paid', 'Outstanding_Amount'}.issubset(clean_df.columns):
    clean_df['Payment_Status_Cleaned'] = clean_df.apply(define_payment_status, axis=1)

clean_df[['Tax_Due', 'Tax_Paid', 'Outstanding_Amount', 'Payment_Status', 'Payment_Status_Cleaned']].head()

## 11. Create Useful Columns for Analysis

These columns are useful for Power BI dashboard and business reporting.

In [ ]:
# Collection percentage
if {'Tax_Paid', 'Tax_Due'}.issubset(clean_df.columns):
    clean_df['Collection_Percentage'] = np.where(
        clean_df['Tax_Due'] > 0,
        (clean_df['Tax_Paid'] / clean_df['Tax_Due']) * 100,
        0
    ).round(2)

# Outstanding flag
if 'Outstanding_Amount' in clean_df.columns:
    clean_df['Outstanding_Flag'] = np.where(clean_df['Outstanding_Amount'] > 0, 'Outstanding', 'No Outstanding')

# Tax range category
if 'Tax_Due' in clean_df.columns:
    clean_df['Tax_Range'] = pd.cut(
        clean_df['Tax_Due'],
        bins=[-1, 5000, 10000, 20000, 50000, np.inf],
        labels=['0-5K', '5K-10K', '10K-20K', '20K-50K', '50K+']
    )

# Property value range
if 'Property_Value' in clean_df.columns:
    clean_df['Property_Value_Range'] = pd.cut(
        clean_df['Property_Value'],
        bins=[-1, 500000, 1000000, 2500000, 5000000, np.inf],
        labels=['Below 5L', '5L-10L', '10L-25L', '25L-50L', '50L+']
    )

clean_df.head()

## 12. Final Quality Check

In [ ]:
print('Final Dataset Shape:', clean_df.shape)
print('
Missing Values After Cleaning:')
print(clean_df.isnull().sum())

print('
Duplicate Rows:', clean_df.duplicated().sum())

In [ ]:
# Final data preview
clean_df.head(10)

## 13. Export Cleaned Dataset

This cleaned dataset can be used directly in Power BI, Tableau, Excel, or Python analysis.

In [ ]:
# Export cleaned dataset
output_file = 'amravati_property_tax_cleaned.csv'
clean_df.to_csv(output_file, index=False)

print(f'Cleaned dataset exported successfully as: {output_file}')

In [ ]:
# Download cleaned dataset in Google Colab
from google.colab import files
files.download('amravati_property_tax_cleaned.csv')

## 14. Basic Analysis After Cleaning

These quick checks help verify that the cleaned data is ready for dashboard creation.

In [ ]:
# Zone-wise tax collection summary
zone_summary = clean_df.groupby('Zone').agg({
    'Tax_Due': 'sum',
    'Tax_Paid': 'sum',
    'Outstanding_Amount': 'sum'
}).reset_index()

zone_summary['Collection_Percentage'] = np.where(
    zone_summary['Tax_Due'] > 0,
    (zone_summary['Tax_Paid'] / zone_summary['Tax_Due']) * 100,
    0
).round(2)

zone_summary

In [ ]:
# Property category-wise summary
category_summary = clean_df.groupby('Property_Category').agg({
    'Property_ID': 'count',
    'Tax_Due': 'sum',
    'Tax_Paid': 'sum',
    'Outstanding_Amount': 'sum'
}).reset_index()

category_summary.rename(columns={'Property_ID': 'Total_Properties'}, inplace=True)
category_summary

## Conclusion

The dataset has been cleaned and prepared for analysis. It is now ready for:

- Power BI Dashboard  
- Tableau Dashboard  
- Excel Reporting  
- Python Analysis  
- GitHub Portfolio Project
